# ライブラリのインポート

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings
warnings.filterwarnings("ignore")
import random
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import torch
from datasets import Dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

# 変数の設定

In [ ]:
class CFG:
    VER = 1
    AUTHOR = "suba"
    COMPETITION = "atmacup17"
    DATA_PATH = Path("/workspace/kaggle_llm_book/ch3/data/takaito/atmacup17")
    MODEL_PATH = "microsoft/deberta-v3-large"
    MODEL_NAME = "deberta-v3-large"
    MAX_LENGTH = 2048
    EPOCHS = 3
    STEPS = 20
    WARMUP_RATIO = 0.05
    TRAIN_BATCH_SPLIT = 8
    TRAIN_BATCH_SIZE = 64
    EVAL_BATCH_SIZE = 64
    LEARNING_RATE = 2e-5
    OPTIM = "adamw_torch"
    USE_GPU = torch.cuda.is_available()
    SEED = 42
    N_SPLIT = 2
    TARGET_COL = "Recommended IND"
    TARGET_COL_CLASS_NUM = 2
    METRIC = "auc"
    METRIC_MAXIMIZE_FLAG = True
    PREFIX  = f"{AUTHOR}_{MODEL_NAME}_seed{SEED}_ver{VER}"
    OUTPUT_DIR = "./model"

# 乱数の設定

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if CFG.USE_GPU:
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
seed_everything(CFG.SEED)

# デバイスの設定

In [ ]:
device = torch.device("cuda" if CFG.USE_GPU else "cpu")
print(device)

# データの読み込み

In [ ]:
clothing_master_df = pd.read_csv(CFG.DATA_PATH / "clothing_master.csv")
train_df = pd.read_csv(CFG.DATA_PATH / "train.csv")
test_df = pd.read_csv(CFG.DATA_PATH / "test.csv")

# データの加工

In [ ]:
def make_text_column(df):
    df["text"] = df["Title"] + " " + df["Review Text"]
    return df

def preprocessing(df, clothing_master_df):
    df["Title"] = df["Title"].fillna("")
    df["Review Text"] = df["Review Text"].fillna("")
    df = df.merge(clothing_master_df, on="Clothing ID", how="left")
    df = make_text_column(df)
    return df

In [ ]:
train_df = preprocessing(train_df, clothing_master_df)
test_df = preprocessing(test_df, clothing_master_df)
train_df["labels"] = train_df[CFG.TARGET_COL].astype(np.int32)

# トークナイザの設定

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_PATH)
def tokenize(sample, tokenizer):
    return tokenizer(sample["text"], max_length=CFG.MAX_LENGTH, truncation=True)

# データセットの作成

In [ ]:
def mk_dataset(df, tokenizer):
    raw_datasets = Dataset.from_pandas(df)
    tokenized_datasets = raw_datasets.map(
        tokenize, 
        fn_kwargs={"tokenizer": tokenizer}
    ).remove_columns(["text"])
    return tokenized_datasets

In [ ]:
ds = mk_dataset(train_df[["text", "labels"]], tokenizer)
ds_test = mk_dataset(test_df[["text"]], tokenizer)

In [ ]:
print(ds)

In [ ]:
print(ds["input_ids"][0])

# モデルの学習と推論

In [ ]:
def compute_metrics(p):
    label_preds, labels = p
    label_prob_preds = torch.softmax(torch.tensor(label_preds), dim = 1).numpy()[:, 1]
    score = roc_auc_score(labels, label_prob_preds)
    return {"auc": score}

In [ ]:
def get_args(CFG, fold):
    args = TrainingArguments(
        output_dir=f"{CFG.OUTPUT_DIR}/{CFG.PREFIX}-fold{fold}",
        report_to="none",
        eval_strategy="steps",
        logging_strategy="steps",
        save_strategy="steps",
        eval_steps=CFG.STEPS,
        logging_steps=CFG.STEPS,
        save_steps=CFG.STEPS,
        save_total_limit=1,
        metric_for_best_model=CFG.METRIC,
        greater_is_better=CFG.METRIC_MAXIMIZE_FLAG,
        optim=CFG.OPTIM,
        learning_rate=CFG.LEARNING_RATE,
        lr_scheduler_type="linear",
        warmup_ratio=CFG.WARMUP_RATIO,
        per_device_train_batch_size=CFG.TRAIN_BATCH_SIZE // CFG.TRAIN_BATCH_SPLIT,
        per_device_eval_batch_size=CFG.EVAL_BATCH_SIZE,
        gradient_accumulation_steps=CFG.TRAIN_BATCH_SPLIT,
        num_train_epochs=CFG.EPOCHS,
        fp16=True,
        seed=CFG.SEED,
        data_seed=CFG.SEED,
    )
    return args

In [ ]:
predictions = np.zeros(len(train_df))
test_predictions = np.zeros(len(test_df))
# 交差検証のために、目的変数が各 fold で均一になるようにデータを分割
kfold = StratifiedKFold(n_splits=CFG.N_SPLIT, shuffle=True, random_state=CFG.SEED)
for fold, (train_index, valid_index) in enumerate(kfold.split(train_df, train_df[CFG.TARGET_COL])):
    # モデルの読み込み & 設定
    config = AutoConfig.from_pretrained(CFG.MODEL_PATH)
    config.num_labels = CFG.TARGET_COL_CLASS_NUM
    model = AutoModelForSequenceClassification.from_pretrained(CFG.MODEL_PATH, config=config)
    trainer = Trainer(
        model=model,
        args=get_args(CFG, fold),
        train_dataset=ds.select(train_index),
        eval_dataset=ds.select(valid_index),
        data_collator=DataCollatorWithPadding(tokenizer),
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )
    # 学習開始
    trainer.train()
    # モデルなどの保存 (Kaggle の出力可能容量が 19.5GB までという制約の関係でコメントアウトしています。)
    trainer.save_model(f"{CFG.OUTPUT_DIR}/{CFG.PREFIX}/{CFG.PREFIX}-fold{fold}")
    tokenizer.save_pretrained(f"{CFG.OUTPUT_DIR}/{CFG.PREFIX}-fold{fold}")
    # 検証セットの推論
    valid_preds = trainer.predict(ds.select(valid_index)).predictions
    predictions[valid_index] = torch.softmax(torch.tensor(valid_preds), dim=1).numpy()[:, 1]
    # テストセットの推論
    test_preds = trainer.predict(ds_test).predictions
    test_predictions += torch.softmax(torch.tensor(test_preds), dim=1).numpy()[:, 1] / CFG.N_SPLIT

In [ ]:
print("CV Score: ", roc_auc_score(train_df[CFG.TARGET_COL], predictions))

In [ ]:
train_df[f"{CFG.PREFIX}_pred_prob"] = predictions
train_df[[f"{CFG.PREFIX}_pred_prob"]].to_csv(f"{CFG.PREFIX}_train_preds.csv", index=False)
test_df[f"{CFG.PREFIX}_pred_prob"] = test_predictions
test_df[[f"{CFG.PREFIX}_pred_prob"]].to_csv(f"{CFG.PREFIX}_test_preds.csv", index=False)

In [ ]:
test_df["target"] = test_df[f"{CFG.PREFIX}_pred_prob"]
test_df[["target"]].to_csv(f"{CFG.PREFIX}_submission.csv", index=False)